# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub

In [2]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

Paste your Hugging Face READ token (hf_...): ··········


In [3]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [4]:
features = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    gsc_sum_position,
    scroll_events
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
AND gsc_data_available IS TRUE
LIMIT 100
""").df()

features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,gsc_sum_position,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,67,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,0,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,616,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,28,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,25,<NA>


In [5]:
features = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
  AND gsc_impressions IS NOT NULL
  AND gsc_clicks IS NOT NULL
  AND gsc_avg_position IS NOT NULL
""").df()


features["ctr"] = (
    features["gsc_clicks"] / features["gsc_impressions"]
).fillna(0)
features["score"] = 0

features.loc[
    (features["gsc_impressions"] >= 100) &
    (features["ctr"] < 0.02),
    "score"
] = 2

features.loc[
    (features["gsc_avg_position"] > 10),
    "score"
] += 1
features["reason_code"] = "NO_ACTION"

features.loc[
    (features["gsc_impressions"] > 100) &
    (features["ctr"] < 0.02),
    "reason_code"
] = "LOW_CTR_HIGH_IMPRESSIONS"

features.loc[
    (features["gsc_avg_position"] > 20),
    "reason_code"
] = "LOW_POSITION"
features["action"] = "No Action"

features.loc[
    features["reason_code"] == "LOW_CTR_HIGH_IMPRESSIONS",
    "action"
] = "Refresh Content"

features.loc[
    features["reason_code"] == "LOW_POSITION",
    "action"
] = "Improve SEO"


features.head(20)



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,score,reason_code,action
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,0.000000,0,NO_ACTION,No Action
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,0.000000,0,NO_ACTION,No Action
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,0.008000,2,LOW_CTR_HIGH_IMPRESSIONS,Refresh Content
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,0.000000,0,NO_ACTION,No Action
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,0.000000,0,NO_ACTION,No Action
5,2026-03-01,client_73cda7b4e4f265ea,content_36c36abc7650d7af,239,1,7.347280,0.004184,2,LOW_CTR_HIGH_IMPRESSIONS,Refresh Content
6,2026-03-01,client_73cda7b4e4f265ea,content_a7da352b73b02668,191,0,7.832461,0.000000,2,LOW_CTR_HIGH_IMPRESSIONS,Refresh Content
7,2026-03-01,client_73cda7b4e4f265ea,content_05434271b257bb68,55,0,3.272727,0.000000,0,NO_ACTION,No Action
8,2026-03-01,client_73cda7b4e4f265ea,content_d056587ff7faca0c,77,0,5.636364,0.000000,0,NO_ACTION,No Action
9,2026-03-01,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2,0,4.500000,0.000000,0,NO_ACTION,No Action


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I will choose **Random Forest** because it can handle different features and capture complex relationships between them.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

We should group the data by client to test whether the model works well on unseen clients.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [11]:

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df["is_declining_label"] = (
    df["trend_direction"].str.lower().eq("down").astype(int)
)

df.head()


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1


In [12]:
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

X = df[features].replace(
    [np.inf, -np.inf], np.nan
).fillna(0)

y = df["is_declining_label"]

print(X.shape)
print(y.shape)

(30000, 6)
(30000,)


In [13]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=df["client_id"])
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Train:", X_train.shape)
print("Test :", X_test.shape)

Train: (23837, 6)
Test : (6163, 6)


In [14]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=6,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=6, random_state=42)

In [15]:
test_scores = model.predict_proba(X_test)[:, 1]

print(test_scores[:10])

[0.56046405 0.49066222 0.67859624 0.7332749  0.50093079 0.57868693
 0.64318492 0.43449178 0.57885389 0.57178867]


In [16]:
test_results = df.iloc[test_idx].copy()

test_results["model_score"] = test_scores

top50_model = test_results.sort_values(
    "model_score",
    ascending=False
).head(50)

print(top50_model["is_declining_label"].mean())

0.6


In [18]:
test_results["hand_rule_score"] = (
    (test_results["impressions_90d"] >= 100).astype(int)
    * (test_results["ctr"] < 0.02).astype(int)
    * test_results["impressions_90d"]
)

top50_hand = test_results.sort_values(
    "hand_rule_score",
    ascending=False
).head(50)

hand_precision = top50_hand["is_declining_label"].mean()

print(f"Hand Rule Precision@50: {hand_precision:.3f}")
top50_hand["is_declining_label"].mean()

Hand Rule Precision@50: 0.720


np.float64(0.72)

In [19]:
rf_precision = top50_model["is_declining_label"].mean()

comparison = pd.DataFrame({
    "Method": ["Hand Rule", "Random Forest"],
    "Precision@50": [hand_precision, rf_precision]
})

comparison

,Method,Precision@50
0,Hand Rule,0.72
1,Random Forest,0.60


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [20]:
test_results["prediction"] = (test_results["model_score"] >= 0.5).astype(int)

errors = test_results[
    test_results["prediction"] != test_results["is_declining_label"]
]

print("Number of errors:", len(errors))
errors[
    ["content_id", "client_id", "is_declining_label", "prediction", "model_score"]
].head(10)

Number of errors: 2682


,content_id,client_id,is_declining_label,prediction,model_score
1,content_a1fb4e703a9e,client_4e07408562,1,0,0.490662
13,content_a5a2fbc76336,client_8527a891e2,0,1,0.733275
23,content_2da6ae9d0882,client_e629fa6598,1,0,0.434492
26,content_72c5c2d73e5a,client_4e07408562,0,1,0.571789
36,content_bce275871a25,client_f369cb89fc,0,1,0.503116
39,content_4595e8704e07,client_8527a891e2,1,0,0.318348
47,content_40cb4af260c0,client_f369cb89fc,1,0,0.470998
51,content_d8a23b5e10c5,client_f369cb89fc,1,0,0.252726
54,content_ff8ea1364b59,client_e629fa6598,1,0,0.443732
56,content_dcebfd222b10,client_f369cb89fc,0,1,0.573896


In [21]:
importance = pd.Series(
    model.feature_importances_,
    index=features
).sort_values(ascending=False)

print(importance)

impressions_90d           0.306507
avg_position              0.261661
content_age_days          0.194865
word_count                0.129151
ctr                       0.064962
days_since_last_update    0.042854
dtype: float64


## 4. Errors and interpretation

The model made 2,682 errors on the test set. The errors included both false negatives, where a declining page was predicted as non-declining, and false positives, where a non-declining page was predicted as declining.

The Random Forest relied most heavily on `impressions_90d`, followed by `avg_position` and `content_age_days`. `word_count`, `ctr`, and `days_since_last_update` had lower feature-importance values in this model.

These results are observed from this test and model configuration. Feature importance shows what the model relied on more heavily; it does not prove that these features cause pages to decline.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.